# Phase 3: Optimized GPU Implementation with cuDNN
**CSC14120 - Parallel Programming**

## Optimizations Applied (vs Phase 2 Naive):

### Core Optimization: cuDNN
- **cuDNN Convolution Forward** - NVIDIA's highly optimized convolution
- **cuDNN Convolution Backward** - Optimized gradient computation
- Automatic algorithm selection for best performance

### Additional Optimizations
1. **Pinned Memory (#5)** - `cudaMallocHost` for faster CPU-GPU transfers
2. **Double Buffering** - Overlap transfer and compute with CUDA streams
3. **Loop Unrolling (#10)** - `#pragma unroll` in custom kernels
4. **Fast Math** - `--use_fast_math` compiler flag

### Expected Speedup
- cuDNN provides 10-50x speedup over naive convolution
- Target: < 10 minutes for 20 epochs

## Huong dan:
1. Zip project (khong bao gom data/)
2. Upload len Colab
3. Chay tat ca cells

In [ ]:
!nvidia-smi
!nvcc --vsersion

In [ ]:
from google.colab import files
import zipfile, os

print("Upload file zip project:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

for root, dirs, _ in os.walk('project'):
    if 'src' in dirs:
        %cd {root}
        break
!ls

In [ ]:
import urllib.request, tarfile
os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 'data/cifar.tar.gz')
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/
print('Done!')

In [ ]:
# Build Phase 3 with cuDNN
# -lcudnn: Link cuDNN library for optimized convolution
# curand is automatically linked for He weight initialization

!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -Iinclude -lcublas -lcudnn \
    -o gpu_train_opt \
    src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp

print('Build complete with cuDNN + He initialization!')

In [ ]:
# Train with cuDNN - should be much faster!
# 20 epochs, batch 64, lr 0.001
!./gpu_train_opt --data data --epochs 20 --batch 64 --lr 0.001 \
    --log phase3_opt.csv --log-txt phase3_opt.txt --save-weights phase3_opt.weights

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('phase3_opt.csv')
ep = df[df['batch'].isna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ep['epoch'], ep['loss'], 'r-o'); ax1.set_title('Training Loss per Epoch'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.grid(True)
ax2.plot(ep['epoch'], ep['epoch_time_sec'], 'orange', marker='o'); ax2.set_title('Time per Epoch'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Time (s)'); ax2.grid(True)
plt.tight_layout(); plt.show()

print("="*50)
print("PHASE 3 OPTIMIZED RESULTS")
print("="*50)
print(f"Best Loss: {ep['best_loss'].iloc[-1]:.6f}")
print(f"Final Loss: {ep['loss'].iloc[-1]:.6f}")
print(f"Avg Time per Epoch: {ep['epoch_time_sec'].mean():.2f}s")
print(f"Total Training Time: {ep['epoch_time_sec'].sum():.2f}s ({ep['epoch_time_sec'].sum()/60:.2f} min)")
print("="*50)

In [ ]:
# Download results
from google.colab import files
files.download('phase3_opt.csv')
files.download('phase3_opt.txt')
files.download('phase3_opt.weights')

In [ ]:
# Show detailed training log
print("="*60)
print("TRAINING LOG (last 20 lines)")
print("="*60)
!tail -30 phase3_opt.txt

## Performance Analysis

### cuDNN Optimizations

| Component | Technique | Expected Speedup |
|:----------|:----------|:-----------------|
| Conv Forward | cuDNN optimized algorithms | 10-50x |
| Conv Backward Data | cuDNN backward data | 10-50x |
| Conv Backward Filter | cuDNN backward filter | 10-50x |
| Bias Backward | cuDNN backward bias | 5-10x |
| Memory Transfer | Pinned memory + streams | 1.5-2x |

### Target Performance
- **Training time for 20 epochs**: < 10 minutes
- **Time per epoch**: ~20-30 seconds
- **Batch processing time**: ~20-40ms